AIML Capstone – Preserving Heritage: Enhancing Tourism with AI
This project has two major components:

Part 1: Deep Learning model to classify historical structures using Transfer Learning

Part 2: Data Science + Collaborative Filtering to build a tourism recommender system

## **Part 1 - Historical Structure Classification with Transfer Learning (TensorFlow/Keras)**

1. Setup and data loading

In [ ]:
# ============================================================
# PART 1 + PART 2 SETUP
# Clean, portable, evaluator-ready
# ============================================================

import os
import gdown

# -----------------------------
# Create base data directories
# -----------------------------
os.makedirs("data/part1", exist_ok=True)
os.makedirs("data/part2", exist_ok=True)

# ============================================================
# PART 1 DATA (Google Drive ZIP)
# ============================================================

file_id = "1qKZGJaKKcqc-WCTobQoEuYu9Dni6IXpg"
url = f"https://drive.google.com/uc?id={file_id}"
output = "part1.zip"

print("Downloading Part 1 dataset...")
gdown.download(url, output, quiet=False)

print("Extracting Part 1 dataset...")
!unzip -o part1.zip -d data/part1/

# ============================================================
# PART 2 DATA (GitHub CSV + XLSX)
# ============================================================

print("Downloading Part 2 datasets...")

!wget -q -O data/part2/user.csv \
  https://raw.githubusercontent.com/banerjeeananya-afk/tourism-capstone/main/data/part2/user.csv

!wget -q -O data/part2/tourism_rating.csv \
  https://raw.githubusercontent.com/banerjeeananya-afk/tourism-capstone/main/data/part2/tourism_rating.csv

!wget -q -O data/part2/tourism_with_id.xlsx \
  https://raw.githubusercontent.com/banerjeeananya-afk/tourism-capstone/main/data/part2/tourism_with_id.xlsx

print("All datasets downloaded and ready.")

# ============================================================
# FINAL TRAIN/TEST PATHS
# ============================================================

train_dir = "/content/data/part1/dataset_hist_structures 2/dataset_hist_structures/Stuctures_Dataset"
test_dir  = "/content/data/part1/dataset_hist_structures 2/dataset_hist_structures/Dataset_test"

print("Train directory:", train_dir)
print("Test directory:", test_dir)


## 1.1 Verify Dataset Structure

In [ ]:
# Verify training directory
!ls "$train_dir"

import os

train_count = sum([len(files) for r, d, files in os.walk(train_dir)])
test_count  = sum([len(files) for r, d, files in os.walk(test_dir)])

print("Train images:", train_count)
print("Test images:", test_count)


## 1.2 Visualize Sample Images per Class

In [ ]:
## Plot Sample Images per Class

import matplotlib.pyplot as plt
import cv2
import os
import random

def show_samples_per_class_clean(base_dir, class_names, samples=8, img_size=(224, 224)):

    rows = len(class_names)
    cols = samples + 1   # 1 title + N images

    plt.figure(figsize=(20, 20))
    idx = 1

    for cls in class_names:
        cls_path = os.path.join(base_dir, cls)

        # Get only valid image files
        images = [img for img in os.listdir(cls_path)
                  if img.lower().endswith(('.jpg', '.jpeg', '.png'))]

        # Randomly sample images
        selected_imgs = random.sample(images, min(samples, len(images)))

        # Add a title cell for the row
        plt.subplot(rows, cols, idx)
        plt.text(0.5, 0.5, cls, fontsize=18, ha='center', va='center')
        plt.axis("off")
        idx += 1

        # Display images
        for img_name in selected_imgs:
            img_path = os.path.join(cls_path, img_name)
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Resize for uniformity
            img = cv2.resize(img, img_size)

            plt.subplot(rows, cols, idx)
            plt.imshow(img)
            plt.axis("off")
            idx += 1

    plt.tight_layout()
    plt.show()


## 1.3 Count Images per Class

In [ ]:
def count_images_per_class(base_dir):
    for cls in sorted(os.listdir(base_dir)):
        cls_path = os.path.join(base_dir, cls)
        if os.path.isdir(cls_path):
            print(f"{cls}: {len(os.listdir(cls_path))} images")

print("Training set:")
count_images_per_class(train_dir)

print("\nTest set:")
count_images_per_class(test_dir)


## 2. Load Dataset into TensorFlow

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory

img_size = (224, 224)
batch_size = 32

train_ds = image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds.class_names
print("Classes:", class_names)

## 3. Optimize Input Pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

## 4. Early Stopping Callback

In [ ]:
class EarlyStopAtValAcc(tf.keras.callbacks.Callback):
    def __init__(self, threshold=0.97):
        super().__init__()
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None):
        val_acc = logs.get("val_accuracy")
        if val_acc is not None and val_acc >= self.threshold:
            print(f"\nReached {self.threshold*100}% validation accuracy. Stopping training.")
            self.model.stop_training = True

early_stop_callback = EarlyStopAtValAcc(threshold=0.97)


## 5. Build EfficientNetB0 Model
### Model A — WITHOUT Augmentation

In [ ]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = tf.keras.applications.efficientnet.preprocess_input(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)

model_no_aug = tf.keras.Model(inputs, outputs)
model_no_aug.summary()


##6. Compile Model A

In [ ]:
model_no_aug.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 7. Train model without Augmentation

In [ ]:
history_no_aug = model_no_aug.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop_callback]
)

## 8. Data Augmentation Layer

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

## 9. Build Model B — WITH Augmentation

In [ ]:
inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)

model_aug = tf.keras.Model(inputs, outputs)

## 10. Compile Model B

In [ ]:
model_aug.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 11. Train Model B

In [ ]:
history_aug = model_aug.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop_callback]
)

## 12. Visualize Training Curves

In [ ]:
def plot_history(history, title):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(len(acc))

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, label='Training Accuracy')
    plt.plot(epochs, val_acc, label='Validation Accuracy')
    plt.title(f"{title} - Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, label='Training Loss')
    plt.plot(epochs, val_loss, label='Validation Loss')
    plt.title(f"{title} - Loss")
    plt.legend()

    plt.show()

plot_history(history_no_aug, "Model Without Augmentation")
plot_history(history_aug, "Model With Augmentation")


## Model Comparison

In [ ]:
import pandas as pd

comparison_df = pd.DataFrame({
    "Model": ["Model A (No Aug)", "Model B (With Aug)"],
    "Train Accuracy": [
        history_no_aug.history['accuracy'][-1],
        history_aug.history['accuracy'][-1]
    ],
    "Val Accuracy": [
        history_no_aug.history['val_accuracy'][-1],
        history_aug.history['val_accuracy'][-1]
    ],
    "Train Loss": [
        history_no_aug.history['loss'][-1],
        history_aug.history['loss'][-1]
    ],
    "Val Loss": [
        history_no_aug.history['val_loss'][-1],
        history_aug.history['val_loss'][-1]
    ]
})

# Apply formatting only to numeric columns
comparison_df.style.format({
    "Train Accuracy": "{:.4f}",
    "Val Accuracy": "{:.4f}",
    "Train Loss": "{:.4f}",
    "Val Loss": "{:.4f}"
})

## Visual Comparison of Models

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

plt.plot(history_no_aug.history['val_accuracy'], label='Model A – Val Accuracy', linewidth=3)
plt.plot(history_aug.history['val_accuracy'], label='Model B – Val Accuracy', linewidth=3)

plt.title("Validation Accuracy Comparison")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

Although data augmentation is typically expected to improve generalization, in this project the baseline model (Model A) achieved slightly higher validation accuracy and lower validation loss than the augmented model (Model B). This is likely due to the large size and natural diversity of the dataset, which already provides sufficient variation for EfficientNetB0 to learn robust features. Additionally, geometric augmentations may distort architectural structures, making classification more challenging. Therefore, Model A is selected as the final model for this task.

## Summary - Part 1

✔Built a deep‑learning classifier for historical architectural structures using EfficientNetB0 and transfer learning.

✔Used a large, diverse dataset of 10,236 training images across multiple heritage categories.

✔Developed and compared two models:

*   Model A: Baseline (no augmentation)
*   Model B: Enhanced with augmentation (flip, rotation, zoom)

✔Implemented an optimized TensorFlow pipeline with caching, shuffling, and prefetching for efficient GPU utilization.

✔Model A outperformed Model B, achieving higher validation accuracy and better generalization.

✔The final model provides a scalable foundation for heritage preservation, automated tagging, and AI‑powered tourism applications.

## **Part 2 — Tourism Data Analysis & Recommender System**

## 1. Load Data

In [ ]:
import pandas as pd

# Local paths (downloaded earlier in Setup section)
path = "/content/data/part2/"

users   = pd.read_csv(path + "user.csv")
ratings = pd.read_csv(path + "tourism_rating.csv")
places  = pd.read_excel(path + "tourism_with_id.xlsx")

users.head(), ratings.head(), places.head()


## 1.1. Perform Preliminary Inspection

In [ ]:
users.info()
ratings.info()
places.info()

print("Missing values in users:")
print(users.isnull().sum())

print("\nMissing values in ratings:")
print(ratings.isnull().sum())

print("\nMissing values in places:")
print(places.isnull().sum())

print("\nDuplicates in users:", users.duplicated().sum())
print("Duplicates in ratings:", ratings.duplicated().sum())
print("Duplicates in places:", places.duplicated().sum())

## 1.2 Clean Data

In [ ]:
# Remove invalid ages
users = users[(users['Age'] > 0) & (users['Age'] < 100)]

# Remove invalid ratings
ratings = ratings[(ratings['Place_Ratings'] >= 1) & (ratings['Place_Ratings'] <= 5)]

# Drop duplicates
users   = users.drop_duplicates()
ratings = ratings.drop_duplicates()
places  = places.drop_duplicates()

## 2. Explore User Demographics

Merge users + ratings

In [ ]:
user_ratings = ratings.merge(users, on='User_Id', how='left')
user_ratings.head()

Age Distribution of tourists

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
sns.histplot(user_ratings['Age'], bins=20, kde=True, color='teal')
plt.title("Age Distribution of Tourists Providing Ratings", fontsize=14)
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

Identify where tourists come from

In [ ]:
plt.figure(figsize=(12,6))
user_ratings['Location'].value_counts().head(10).plot(kind='bar', color='coral')
plt.title("Top 10 Cities Tourists Come From", fontsize=14)
plt.xlabel("City")
plt.ylabel("Number of Tourists")
plt.xticks(rotation=45)
plt.show()

## 3. Explore Tourist Spots & Categories

## Categories

In [ ]:
places['Category'].value_counts()

## Dominant categorie per city


In [ ]:
location_category = (
    places.groupby(['City', 'Category'])
          .size()
          .unstack(fill_value=0)
)

dominant_category = location_category.idxmax(axis=1)
dominant_category.head(10)

## Best Cities for Nature Lover

In [ ]:
nature_spots = places[places['Category'].str.contains("Cagar Alam", case=False, na=False)]
nature_spots['City'].value_counts().head(10)

## 4. Most Loved Tourist Spots, Cities, and Categories

## Merge RAting with Places

In [ ]:
df = ratings.merge(places, on='Place_Id', how='left')
df.head()

## Most loved Tourist Spots

In [ ]:
spot_scores = (
    df.groupby('Place_Name')['Place_Ratings']
      .mean()
      .sort_values(ascending=False)
)

spot_scores.head(10)

## Most Loved Cities

In [ ]:
city_scores = (
    df.groupby('City')['Place_Ratings']
      .mean()
      .sort_values(ascending=False)
)

city_scores.head(10)

## Most Loved Categories

In [ ]:
category_scores = (
    df.groupby('Category')['Rating']
      .mean()
      .sort_values(ascending=False)
)
category_scores

## 5. Collaborative Filtering Recommender System

## User–Item Matrix

In [ ]:
user_item_matrix = df.pivot_table(
    index='User_Id',
    columns='Place_Name',
    values='Rating'
)
user_item_matrix.head()

## Item - Item Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Fill NaNs with 0 for similarity computation
item_matrix = user_item_matrix.fillna(0).T  # places as rows

similarity_matrix = cosine_similarity(item_matrix)
item_similarity_df = pd.DataFrame(
    similarity_matrix,
    index=item_matrix.index,
    columns=item_matrix.index
)
item_similarity_df.head()

## Recommendation Function

In [ ]:
def recommend_places(place_name, n=5):
    if place_name not in item_similarity_df.index:
        print("Place not found in database.")
        return []

    # Get similarity scores for the given place
    sim_scores = item_similarity_df[place_name].sort_values(ascending=False)

    # Drop the place itself
    sim_scores = sim_scores.drop(place_name)

    # Return top n similar places
    return sim_scores.head(n)

In [ ]:
recommend_places("Pulau Pari", n=5)

## Summary — Part 2
✔ Cleaned and validated all datasets

✔ Explored demographics, tourist origins, and category insights

✔ Identified most loved cities, categories, and tourist spots

✔ Built a collaborative filtering recommender system

✔ Generated meaningful recommendations (e.g., for Pulau Pari)